# Multi-Agent AI Analyst — deploy from Colab (F14)

The fastest way to a **public URL with no credit card**. Colab gives ~12 GB RAM,
so everything fits comfortably.

**Before you start:** zip your `backend/` folder (without `.venv`, `qdrant_data`
and `.env`) and have your Gemini key ready.

Run the cells top to bottom. The last one prints a public link.

## 1. Upload the backend

Pick your `backend.zip` when the file dialog appears.

In [ ]:
from google.colab import files
import zipfile, os, pathlib

uploaded = files.upload()          # choose backend.zip
name = next(iter(uploaded))
with zipfile.ZipFile(name) as z:
    z.extractall('/content')

# Find the folder that actually contains main.py — zip layouts vary.
root = next(p.parent for p in pathlib.Path('/content').rglob('main.py')
            if 'site-packages' not in str(p))
os.chdir(root)
print('backend is at:', root)
print(sorted(os.listdir())[:15])

## 2. Install dependencies

Takes 2–3 minutes.

In [ ]:
!pip install -q -r requirements.txt pyngrok

## 3. Your API key

Uses Colab's **Secrets** panel (the 🔑 icon on the left) so the key never
appears in the notebook — important, because a shared notebook would otherwise
leak it.

Add a secret named `GEMINI_API_KEY`, paste the **`sk-…` key your class issued for
the LiteLLM proxy**, and enable notebook access. This is not a Google AI Studio
key — the proxy holds the upstream Google credentials.

> The model names below must be the ones the proxy exposes
> (`gemini-flash-lite`, `gemini-embedding`). Google's own model ids such as
> `gemini-3.1-flash-lite` or `models/gemini-embedding-001` do **not** exist there
> and `verify_key.py` will fail on them. If your key is authorised for
> `gemini-flash`, set `REASONING_MODEL=gemini-flash` for a stronger critic.

In [ ]:
from google.colab import userdata

# Accept either secret name so an older notebook setup still works.
key = userdata.get('GEMINI_API_KEY') or userdata.get('GOOGLE_API_KEY')
assert key, 'Add GEMINI_API_KEY in the Secrets panel (key icon, left sidebar)'

with open('.env', 'w', encoding='utf-8') as f:
    f.write(f'GEMINI_API_KEY={key}\n')
    # Model names as exposed by the class LiteLLM proxy — not Google's own ids.
    f.write('LLM_MODEL=gemini-flash-lite\n')
    f.write('REASONING_MODEL=gemini-flash-lite\n')
    f.write('EMBEDDING_MODEL=gemini-embedding\n')
    f.write('EMBEDDING_DIM=3072\n')

# Confirms the key works AND that it is authorised for each model above.
!python verify_key.py

## 4. Build the database and index the documents

In [ ]:
!python data/seed_db.py
!python ingest.py --reset

## 5. Sanity check before going public

In [ ]:
!python smoke_test.py

## 6. Start the API and get a public URL

**One worker only.** Embedded Qdrant is a single-process store — a second
worker cannot open the same folder and will crash on startup.

In [ ]:
import subprocess, time, requests

server = subprocess.Popen(
    ['uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000', '--workers', '1'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for _ in range(30):
    time.sleep(2)
    try:
        health = requests.get('http://localhost:8000/health', timeout=3).json()
        print('backend up:', health)
        break
    except Exception:
        pass
else:
    raise RuntimeError('server did not start — check the log below')

In [ ]:
# ngrok needs a free authtoken: https://dashboard.ngrok.com/get-started/your-authtoken
# Add it as a Colab secret named NGROK_TOKEN. Still free, still no card.
from google.colab import userdata
from pyngrok import ngrok

ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))
public_url = ngrok.connect(8000).public_url

print('\n' + '=' * 62)
print('PUBLIC BACKEND URL:', public_url)
print('=' * 62)
print('\nPaste it into the "Backend" box at the top of index.html.')
print('Put this URL in your README as the F14 deliverable.')

## 7. Prove it works

Asks a question that needs two specialists, straight through the public URL.

In [ ]:
import requests, json

r = requests.post(f'{public_url}/ask', timeout=180, json={
    'question': 'How many customers churned in Q3 2024, and what does the '
                'postmortem say caused it?'})
data = r.json()

print('AGENT PATH')
for i, step in enumerate(data['steps'], 1):
    print(f'  {i:2}. {step}')
print('\nANSWER\n' + data['answer'])

---

**Keep this tab open** — the link dies when the Colab runtime stops (~72 h max,
sooner if idle). For an always-on deployment use `render.yaml` instead.

Screenshot the agent path above and the frontend connected to this URL: that is
the F14 evidence.